# EMI Poles: Activation-Space Evidence & Intuition Prototypes

Produces the **mean-centered evidence and intuition pole vectors in raw activation space** (768-dim BERT / 1024-dim GPT-2) used for downstream EMI scoring without an SAE.

Pipeline:
1. Load the Phase-1 exemplar activations from `sparse_sae_emi_pipeline.ipynb` (`*_phase1_activations.npy`, shape `(200000, dim)`, evidence first then intuition) and average each pole to get the raw activation prototypes.
2. Compute the mean activation across 1M speeches (streamed) and **save it immediately**.
3. **Reload** the saved 1M mean and subtract it from each prototype to get the centered evidence/intuition poles, then save those.

This is the activation-space analogue of the SAE-feature pipeline — there is **no SAE encode step**, so a downstream service can score text by comparing the raw mean-pooled activation directly against these poles. File paths match `emi_analysis.ipynb`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [9]:
import os
import numpy as np

RESULTS_DIR     = './../outputs/sparse_sae_emi'
ACTIVATIONS_DIR = './../outputs/activations'

N_EV       = 100_000   # evidence exemplars come first (must match PHASE1_TOP_N_PER_POLE)
CHUNK_SIZE = 100_000   # rows processed at once when streaming the 1M mean

# Phase-1 exemplar activations (from sparse_sae_emi_pipeline.ipynb)
BERT_P1_ACT_PATH = f'{RESULTS_DIR}/bert_phase1_activations.npy'
GPT2_P1_ACT_PATH = f'{RESULTS_DIR}/gpt2_phase1_activations.npy'

# 1M activation files (from collect_activations_bert/gpt2.py)
BERT_1M_PATH = f'{ACTIVATIONS_DIR}/bert_last_1000000.npy'
GPT2_1M_PATH = f'{ACTIVATIONS_DIR}/gpt2_last_1000000.npy'

for name, path in [
    ('BERT phase-1 acts', BERT_P1_ACT_PATH), ('GPT-2 phase-1 acts', GPT2_P1_ACT_PATH),
    ('BERT 1M acts',      BERT_1M_PATH),     ('GPT-2 1M acts',      GPT2_1M_PATH),
]:
    print(f'  {name:<20} exists={os.path.isfile(path)}  {path}')

  BERT phase-1 acts    exists=False  ./../outputs/sparse_sae_emi/bert_phase1_activations.npy
  GPT-2 phase-1 acts   exists=False  ./../outputs/sparse_sae_emi/gpt2_phase1_activations.npy
  BERT 1M acts         exists=True  ./../outputs/activations/bert_last_1000000.npy
  GPT-2 1M acts        exists=True  ./../outputs/activations/gpt2_last_1000000.npy


## Section 1 — Build activation-space prototypes from exemplar activations

Average the raw mean-pooled activations of each pole's exemplars. The Phase-1 array is ordered evidence-first (`[:N_EV]`) then intuition (`[N_EV:]`), matching the order used when the activations were collected.

In [11]:
def build_prototypes(p1_path, n_ev, label):
    acts = np.load(p1_path)   # (200000, dim)
    ev   = acts[:n_ev].mean(axis=0).astype(np.float32)
    iv   = acts[n_ev:].mean(axis=0).astype(np.float32)
    print(f'{label}: phase-1 acts {acts.shape}  ->  evidence {ev.shape}, intuition {iv.shape}')
    return ev, iv

bert_ev_proto, bert_in_proto = build_prototypes(BERT_P1_ACT_PATH, N_EV, 'BERT')
gpt2_ev_proto, gpt2_in_proto = build_prototypes(GPT2_P1_ACT_PATH, N_EV, 'GPT-2')

BERT: phase-1 acts (200000, 768)  ->  evidence (768,), intuition (768,)
GPT-2: phase-1 acts (200000, 1024)  ->  evidence (1024,), intuition (1024,)


## Section 2 — Compute & save the 1M-Speech activation mean

Stream through the 1M raw activations in chunks, accumulating a running sum (no model, no SAE — just averaging). The mean is saved to disk immediately after computation (skip-if-exists).

In [12]:
BERT_1M_MEAN_PATH = f'{RESULTS_DIR}/bert_1m_activation_mean.npy'
GPT2_1M_MEAN_PATH = f'{RESULTS_DIR}/gpt2_1m_activation_mean.npy'

def compute_1m_activation_mean(acts_path, chunk_size, label):
    """Memory-mapped streaming mean over the 1M activations. Returns (dim,) float32."""
    acts = np.load(acts_path, mmap_mode='r')   # ~3-4 GB on disk, not in RAM
    n, dim = acts.shape
    print(f'{label}: {n:,} activations  shape={acts.shape}')

    running_sum = np.zeros(dim, dtype=np.float64)
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        running_sum += acts[start:end].astype(np.float64).sum(axis=0)
        if (start // chunk_size) % 2 == 0:
            print(f'  {end:,}/{n:,}')

    mean_vec = (running_sum / n).astype(np.float32)
    print(f'{label} mean activation: shape={mean_vec.shape}  norm={np.linalg.norm(mean_vec):.4f}')
    return mean_vec


if os.path.isfile(BERT_1M_MEAN_PATH):
    bert_1m_mean = np.load(BERT_1M_MEAN_PATH)
    print(f'Loaded cached BERT 1M mean: {bert_1m_mean.shape}')
else:
    print('Computing BERT 1M activation mean...')
    bert_1m_mean = compute_1m_activation_mean(BERT_1M_PATH, CHUNK_SIZE, 'BERT')
    np.save(BERT_1M_MEAN_PATH, bert_1m_mean)
    print(f'Saved: {BERT_1M_MEAN_PATH}')

if os.path.isfile(GPT2_1M_MEAN_PATH):
    gpt2_1m_mean = np.load(GPT2_1M_MEAN_PATH)
    print(f'Loaded cached GPT-2 1M mean: {gpt2_1m_mean.shape}')
else:
    print('Computing GPT-2 1M activation mean...')
    gpt2_1m_mean = compute_1m_activation_mean(GPT2_1M_PATH, CHUNK_SIZE, 'GPT-2')
    np.save(GPT2_1M_MEAN_PATH, gpt2_1m_mean)
    print(f'Saved: {GPT2_1M_MEAN_PATH}')

Computing BERT 1M activation mean...
BERT: 1,000,000 activations  shape=(1000000, 768)
  100,000/1,000,000
  300,000/1,000,000
  500,000/1,000,000
  700,000/1,000,000
  900,000/1,000,000
BERT mean activation: shape=(768,)  norm=7.1201
Saved: ./../outputs/sparse_sae_emi/bert_1m_activation_mean.npy
Computing GPT-2 1M activation mean...
GPT-2: 1,000,000 activations  shape=(1000000, 1024)
  100,000/1,000,000
  300,000/1,000,000
  500,000/1,000,000
  700,000/1,000,000
  900,000/1,000,000
GPT-2 mean activation: shape=(1024,)  norm=53.1479
Saved: ./../outputs/sparse_sae_emi/gpt2_1m_activation_mean.npy


## Section 3 — Reload means and build centered poles

Reload the just-saved 1M means from disk, then subtract them from each prototype to form the centered evidence and intuition poles. The cosine check confirms centering improves pole separation (lower = better).

In [13]:
# Reload the saved 1M means, then center each pole
bert_1m_mean = np.load(BERT_1M_MEAN_PATH)
gpt2_1m_mean = np.load(GPT2_1M_MEAN_PATH)

bert_ev_centered = bert_ev_proto - bert_1m_mean
bert_in_centered = bert_in_proto - bert_1m_mean
gpt2_ev_centered = gpt2_ev_proto - gpt2_1m_mean
gpt2_in_centered = gpt2_in_proto - gpt2_1m_mean

def proto_cosine(a, b):
    return float((a / np.linalg.norm(a)) @ (b / np.linalg.norm(b)))

print('Prototype cosine similarity (lower = better separation):')
print(f'  BERT  original:  {proto_cosine(bert_ev_proto, bert_in_proto):+.4f}'
      f'   centered: {proto_cosine(bert_ev_centered, bert_in_centered):+.4f}')
print(f'  GPT-2 original:  {proto_cosine(gpt2_ev_proto, gpt2_in_proto):+.4f}'
      f'   centered: {proto_cosine(gpt2_ev_centered, gpt2_in_centered):+.4f}')

Prototype cosine similarity (lower = better separation):
  BERT  original:  +0.8980   centered: -0.3877
  GPT-2 original:  +0.9663   centered: -0.2778


In [14]:
# ── Save centered poles (activation space: 768 BERT / 1024 GPT-2) ──────────
np.save(f'{RESULTS_DIR}/bert_evidence_prototype_centered.npy',  bert_ev_centered)
np.save(f'{RESULTS_DIR}/bert_intuition_prototype_centered.npy', bert_in_centered)
np.save(f'{RESULTS_DIR}/gpt2_evidence_prototype_centered.npy',  gpt2_ev_centered)
np.save(f'{RESULTS_DIR}/gpt2_intuition_prototype_centered.npy', gpt2_in_centered)

print('Outputs saved:')
for fname, vec in [
    ('bert_1m_activation_mean.npy',            bert_1m_mean),
    ('gpt2_1m_activation_mean.npy',            gpt2_1m_mean),
    ('bert_evidence_prototype_centered.npy',   bert_ev_centered),
    ('bert_intuition_prototype_centered.npy',  bert_in_centered),
    ('gpt2_evidence_prototype_centered.npy',   gpt2_ev_centered),
    ('gpt2_intuition_prototype_centered.npy',  gpt2_in_centered),
]:
    path = f'{RESULTS_DIR}/{fname}'
    if os.path.isfile(path):
        print(f'  {fname:<44} dim={vec.shape[0]:<5} ({os.path.getsize(path)/1e3:.1f} KB)')

Outputs saved:
  bert_1m_activation_mean.npy                  dim=768   (3.2 KB)
  gpt2_1m_activation_mean.npy                  dim=1024  (4.2 KB)
  bert_evidence_prototype_centered.npy         dim=768   (3.2 KB)
  bert_intuition_prototype_centered.npy        dim=768   (3.2 KB)
  gpt2_evidence_prototype_centered.npy         dim=1024  (4.2 KB)
  gpt2_intuition_prototype_centered.npy        dim=1024  (4.2 KB)
